In [ ]:
import pandas as pd

# ========================
# FILE PATHS (adjust if files are in different folders)
# ========================
mapping_file = "main_with_subs_only.xlsx"          # or "D:/Tushar/main_with_subs_only.xlsx"
indent_file  = "Monthly Indent.xlsx"               # or full path

# ========================
# LOAD FILES
# ========================
df_mapping = pd.read_excel(mapping_file)
df_indent  = pd.read_excel(indent_file, sheet_name="Sheet1")

# ========================
# RENAME COLUMNS
# ========================
df_mapping = df_mapping.rename(columns={
    'Main_Label':   'Child_Part',
    'Sub_Label':    'Switch_Part',
    'Main_Count':   'Historical_Child_Count',   # ignored in calculation
    'Sub_Count':    'Qty_per_Switch'            # ← this is qty of child per switch
})

df_indent = df_indent.rename(columns={
    'Part number':  'Switch_Part'
})

# ========================
# MONTH COLUMNS
# ========================
month_cols = ["Feb'26", "Mar'26", "Apr'26", "May'26", "Jun'26", "Jul'26"]

# Clean names for output columns (Feb'26 → Feb26)
clean_months = [m.replace("'", "") for m in month_cols]

# ========================
# MERGE
# ========================
df_merged = pd.merge(
    df_mapping[['Child_Part', 'Switch_Part', 'Qty_per_Switch']],
    df_indent[['Switch_Part'] + month_cols],
    on='Switch_Part',
    how='left'
)

print(f"Rows after merge: {len(df_merged)}")
if month_cols:
    missing = df_merged[month_cols[0]].isna().sum()
    print(f"Missing monthly data in first month: {missing} rows (0 is ideal)")

# ========================
# CALCULATE DAILY & 2-DAYS REQUIREMENTS
# ========================
for month, clean in zip(month_cols, clean_months):
    daily_col   = f"Daily_{clean}"
    twodays_col = f"2Days_{clean}"
    
    # Daily switch demand = monthly indent / 30
    df_merged[daily_col]   = (df_merged[month] / 30.0).round(2)
    
    # 2-days child requirement from this switch = daily_demand × qty_per_switch × 2
    df_merged[twodays_col] = (df_merged[daily_col] * df_merged['Qty_per_Switch'] * 2).round(2)

# ========================
# 1. TOTALS PER CHILD PART (sum across all switches)
# ========================
agg_dict = {f"Daily_{clean}": 'sum' for clean in clean_months}

totals = df_merged.groupby('Child_Part', as_index=False).agg(agg_dict)

for clean in clean_months:
    totals[f"2Days_{clean}"] = (totals[f"Daily_{clean}"] * 2).round(2)

# Column order
totals = totals[
    ['Child_Part'] +
    [f"Daily_{clean}" for clean in clean_months] +
    [f"2Days_{clean}" for clean in clean_months]
]

totals.to_excel("Child_Totals_2Days_Per_Month.xlsx", index=False)
print(f"Totals saved → Child_Totals_2Days_Per_Month.xlsx ({len(totals)} rows)")

# ========================
# 2. DETAILED BREAKDOWN (per child-switch combination)
# ========================
detailed_cols = (
    ['Child_Part', 'Switch_Part', 'Qty_per_Switch'] +
    month_cols +                           # original monthly indents
    [f"Daily_{clean}" for clean in clean_months] +
    [f"2Days_{clean}" for clean in clean_months]
)

detailed = df_merged[detailed_cols]

detailed.to_excel("Child_Detailed_Breakdown_2Days.xlsx", index=False)
print(f"Detailed saved → Child_Detailed_Breakdown_2Days.xlsx ({len(detailed)} rows)")

print("\nDone! Open the two new files.")